# Scraping

## Data Collection Process

### Import Libraries

In [ ]:
%pip install --quiet BeautifulSoup4 requests
!pip install --upgrade numpy
!pip install --upgrade gensim
! pip install pyLDAvis

In [ ]:
import pandas as pd # data manipulation and storage
import requests  # To make HTTP requests and retrieve web page content
from bs4 import BeautifulSoup  # To extract data and parse HTML content
import time # To pause between requests
import os # To check file paths and file existence
from typing import List  # For type hints
from textblob import TextBlob  # For sentiment analysis
from wordcloud import WordCloud # to Visualise the results
import string  # Import the string module for punctuation
import re  # For module for regular expressions
import matplotlib.pyplot as plt # To create visualisations
import seaborn as sns # To create visualisations
import nltk # For natural language processing tasks
from nltk.stem import WordNetLemmatizer, PorterStemmer # For stemming and lemmatization
from nltk.corpus import stopwords # For removing stop words
from nltk.tokenize import TweetTokenizer # For tokenizing text
nltk.download('stopwords')
nltk.download('wordnet')
import numpy as np # For numerical operations
import gensim # For topic modeling
from gensim import corpora # For creating a corpus
import pyLDAvis # For interactive topic model visualisation
import pyLDAvis.gensim_models # For using pyLDAvis with gensim models

### Airline Reviews Setup

In [ ]:
## List of specific airlines
airlines = [ "British Airways", "Lufthansa", "Emirates", "Qatar", "Singapore Airlines",
            "American Airlines" ]

In [ ]:
# Defining the url of the site
base_url = "https://www.airlinequality.com/airline-reviews/{airline}/"

### Download HTML files for each airline

In [ ]:
# Download and save HTML files for individual airline review pages
for airline in airlines:

    formatted_airline = airline.lower().replace(" ", "-")

    url = base_url.replace("{airline}", formatted_airline)

    response = requests.get(url)
    if response.status_code == 200:
        print(f"Downloading HTML for {airline}...")

        with open(f"{formatted_airline}.html", "w", encoding="utf-8") as file:
            file.write(response.text)
        print(f"HTML file saved for {airline}")
    else:
        print(f"Failed to retrieve HTML for {airline} (Status code: {response.status_code})")

## Data Transformation

### Scrape Reviews from Downloaded Files

In [ ]:
# Scrape Reviews from Downloaded Files
max_reviews = 20
all_reviews = []

for airline in airlines:
    formatted_airline = airline.lower().replace(" ", "-")
    file_path = f"{formatted_airline}.html"

    if os.path.exists(file_path):
        print(f"Extracting reviews for {airline}...")
        reviews_temp = []

        with open(file_path, "r", encoding="utf-8") as file:
            soup = BeautifulSoup(file, "html.parser")
            articles = soup.find_all("article", class_="comp_media-review-rated")

            for article in articles[:max_reviews]:
                time.sleep(5)

                title_element = article.find("h2")
                title = title_element.text if title_element else "No Title"

                rating_element = article.find("div", class_="rating-10")
                rating = rating_element.text.strip()[0] if rating_element and rating_element.text.strip() else "No Rating"

                review_text_element = article.find("div", class_="text_content")
                review_text = review_text_element.text if review_text_element else "No Review"

                reviews_temp.append([airline, title, rating, review_text])

        print(f"Found {len(reviews_temp)} reviews for {airline}.")
        all_reviews.extend(reviews_temp)
    else:
        print(f"File not found for {airline}.")

print(f"Total reviews consolidated: {len(all_reviews)}")

## Data Storage

### Save all reviews to a single file

In [ ]:
# Save reviews to a file
filename = "all_reviews.txt"

with open(filename, "w", encoding="utf-8") as file:
    file.write("Airline Name | Title | Rating | Verified | Review_Text\n")
    file.write("\n")

    for review in all_reviews:
        if len(review) == 4:
            airline, title, rating, review_text = review

            split_review = review_text.split("|", 1)

            if len(split_review) == 2:
                verified, review_text = split_review
                verified = verified.strip()
                review_text = review_text.strip()
            else:
                verified = "Not Verified"

        else:
            print(f"Skipping review with unexpected format: {review}")
            continue

        file.write(f"{airline} | {title} | {rating} | {verified} | {review_text}\n")

print(f"All reviews saved to '{filename}'")

### Read and Print the saved reviews

In [ ]:
# Reads the saved reviews file and prints the reviews
with open("all_reviews.txt", "r", encoding="utf-8") as file:
    content = file.read()
    print(content)

# Natural Language Processing

## Loading the File

In [ ]:
# Load the reviews data
df = pd.read_csv("all_reviews.txt", sep="|", encoding="utf-8", skiprows=[1])
df.columns = df.columns.str.strip()

print(df.head())

## Text Cleaning

In [ ]:
# Take the first review for verification
Review_Text = df['Review_Text'][0]

In [ ]:
# Remove Stop Words
STOPWORDS = set(stopwords.words('english'))
no_stopwords = " ".join([word for word in review_text.split() if word.lower() not in STOPWORDS])

In [ ]:
# Remove Punctuation
# This step operates on the 'no_stopwords' text from the previous step
no_punct = no_stopwords.translate(str.maketrans('', '', string.punctuation))

In [ ]:
# Remove URLs
# This step operates on the 'no_punct' text from the previous step
no_urls = re.sub(r'https?://\S+|www\.\S+', '', no_punct)

In [ ]:
# Remove Numbers
# This step operates on the 'no_urls' text from the previous step
no_numbers = re.sub(r'\d+', '', no_urls)

In [ ]:
# Remove Short Words
# This step operates on the 'no_numbers' text from the previous step
no_short_words = " ".join([word for word in no_numbers.split() if len(word) > 2])

## Text preprocessing

In [ ]:
# Tokenize Text
tokenizer = TweetTokenizer()
tokens = tokenizer.tokenize(no_short_words)

In [ ]:
# Apply Stemming
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in tokens]

In [ ]:
# Apply Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]

## Sentiment Analysis

In [ ]:
# Get Subjectivity and Polarity Score
df['Subjectivity'] = df['Review_Text'].apply(lambda x: TextBlob(x).sentiment.subjectivity)
df['Polarity'] = df['Review_Text'].apply(lambda x: TextBlob(x).sentiment.polarity)

print("DataFrame with Sentiment Scores:")
print(df[['Review_Text', 'Subjectivity', 'Polarity']].head())

In [ ]:
# Get Sentiment Label
df['Sentiment'] = df['Polarity'].apply(lambda score: 'Negative' if score < 0 else ('Neutral' if score == 0 else 'Positive'))

print("DataFrame with Sentiment Labels:")
print(df[['Review_Text', 'Polarity', 'Sentiment']].head())

In [ ]:
# Calculate and Display Sentiment Percentages
# Create DataFrames for Positive, Negative, and Neutral Reviews before calculating percentages
df_positive = df[df['Sentiment'] == 'Positive']
df_negative = df[df['Sentiment'] == 'Negative']
df_neutral = df[df['Sentiment'] == 'Neutral']

positive_percentage = round((len(df_positive) / len(df)) * 100, 1)
negative_percentage = round((len(df_negative) / len(df)) * 100, 1)
neutral_percentage = round((len(df_neutral) / len(df)) * 100, 1)

# Print the percentages
print(f"Positive Reviews: {positive_percentage}%")
print(f"Negative Reviews: {negative_percentage}%")
print(f"Neutral Reviews: {neutral_percentage}%")

In [ ]:
# Analyse Verification Status
total_reviews = len(df)
verified_reviews = df['Verified'].value_counts().get('✅ Trip Verified', 0)
non_verified_reviews = df['Verified'].value_counts().get('Not Verified', 0)
verified_percentage = round((verified_reviews / total_reviews) * 100, 1)
non_verified_percentage = round((non_verified_reviews / total_reviews) * 100, 1)

print(f"\nVerification Status Distribution:")
print(f"Total Reviews: {total_reviews}")
print(f"Verified Reviews: {verified_reviews} ({verified_percentage}%)")
print(f"Non-Verified Reviews: {non_verified_reviews} ({non_verified_percentage}%)")

negative_reviews = df[df['Sentiment'] == 'Negative']
negative_non_verified = negative_reviews[negative_reviews['Verified'] == 'Not Verified']
negative_non_verified_percentage = round((len(negative_non_verified) / len(negative_reviews)) * 100, 1) if len(negative_reviews) > 0 else 0

print(f"\nNon-Verified Reviews Among Negative Sentiment:")
print(f"Total Negative Reviews: {len(negative_reviews)}")
print(f"Non-Verified Negative Reviews: {len(negative_non_verified)} ({negative_non_verified_percentage}%)")

## Feature Engineering

In [ ]:
# Analyse Delays
delay_keywords = ["delay", "delayed", "late", "cancellation", "cancelled"]
df_negative = df[df['Sentiment'] == 'Negative']
airline_keyword_counts = {}

for _, row in df_negative.iterrows():
    airline = row['Airline Name']
    review_text = row['Review_Text']
    keyword_count = sum(1 for keyword in delay_keywords if keyword in review_text.lower().split())
    airline_keyword_counts[airline] = airline_keyword_counts.get(airline, 0) + keyword_count

result_df = pd.DataFrame(list(airline_keyword_counts.items()), columns=['Airline', 'Delay_Keyword_Count'])

print(result_df)

## Visualisations

In [ ]:
# Create DataFrames for Positive, Negative, and Neutral Reviews
df_negative = df[df['Sentiment'] == 'Negative']
df_positive = df[df['Sentiment'] == 'Positive']
df_neutral = df[df['Sentiment'] == 'Neutral']

# For Negative Reviews
reviews_text_negative = ' '.join(df_negative['Review_Text'])
wordcloud_negative = WordCloud(width=800, height=500, random_state=42, max_font_size=100).generate(reviews_text_negative)
plt.figure(figsize=(15, 8))
plt.imshow(wordcloud_negative, interpolation='bilinear')
plt.axis('off')
plt.title("Negative Reviews Word Cloud")
plt.show()

In [ ]:
# Plot Polarity vs Subjectivity
plt.figure(figsize=(8, 6))
plt.scatter(df['Polarity'], df['Subjectivity'], color='Blue')
plt.title('Sentiment Analysis (Polarity vs Subjectivity)')
plt.xlabel('Polarity')
plt.ylabel('Subjectivity')
plt.show()

In [ ]:
# Plot Sentiment Counts
plt.title('Sentiment Analysis')
plt.xlabel('Sentiment')
plt.ylabel('Count')
df['Sentiment'].value_counts().plot(kind='bar')
plt.show()

In [ ]:
# Sentiment vs. Airline
plt.figure(figsize=(10, 6))
sns.boxplot(x='Airline Name', y='Polarity', data=df)
plt.title('Sentiment Distribution by Airline')
plt.xlabel('Airline')
plt.ylabel('Sentiment Polarity')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# Overall vs. Negative Average Ratings by Airline
average_ratings = df.groupby('Airline Name')['Rating'].apply(lambda x: int(pd.Series(x).astype(int).mean())).reset_index()
average_ratings['Rating_Type'] = 'Overall Average'

negative_reviews = df[df['Sentiment'] == 'Negative']

average_ratings_negative = negative_reviews.groupby('Airline Name')['Rating'].apply(lambda x: int(pd.Series(x).astype(int).mean())).reset_index()
average_ratings_negative['Rating_Type'] = 'Negative Average'

all_ratings = pd.concat([average_ratings, average_ratings_negative])

plt.figure(figsize=(12, 6))
sns.barplot(x='Airline Name', y='Rating', hue='Rating_Type', data=all_ratings, palette={'Overall Average': 'blue', 'Negative Average': 'slategrey'})
plt.title('Average Rating vs. Negative Average Rating by Airline')
plt.xlabel('Airline')
plt.ylabel('Average Rating')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Rating Type')
plt.show()

In [ ]:
# Create a bar plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Airline', y='Delay_Keyword_Count', data=result_df, color='blue')
plt.title('Frequency of Delay-Related Keywords by Airline')
plt.xlabel('Airline')
plt.ylabel('Count of Delay Keywords')
plt.xticks(rotation=45, ha='right')
plt.show()

# Topic Modelling

In [ ]:
# Topic Modelling Using Latent Dirichlet Allocation (LDA).
# Preprocess for Topic Modeling
processed_reviews = df['Review_Text'].tolist()

# Lowercase
processed_reviews = [review.lower() for review in processed_reviews]

# Remove punctuation
processed_reviews = [review.translate(str.maketrans('', '', string.punctuation)) for review in processed_reviews]

# Remove URLs and numbers
processed_reviews = [re.sub(r'https?://\S+|www\.\S+', '', review) for review in processed_reviews]
processed_reviews = [re.sub(r'\d+', '', review) for review in processed_reviews]

# Tokenize
processed_reviews = [review.split() for review in processed_reviews]

# Remove stop words
stop_words = set(stopwords.words('english'))
processed_reviews = [[word for word in review if word not in stop_words] for review in processed_reviews]

# Lemmatize
lemmatizer = WordNetLemmatizer()
processed_reviews = [[lemmatizer.lemmatize(word) for word in review] for review in processed_reviews]

# Remove short words
processed_reviews = [[word for word in review if len(word) > 2] for review in processed_reviews]

# LDA
dictionary = corpora.Dictionary(processed_reviews)
corpus = [dictionary.doc2bow(review) for review in processed_reviews]

In [ ]:
# Perform LDA Topic Modeling and Display Topics
num_topics = 5
lda_model = gensim.models.LdaMulticore(corpus, num_topics=num_topics,
                                      id2word=dictionary, passes=10,
                                      workers=2)

topics = lda_model.show_topics(num_topics=num_topics, num_words=5, formatted=False)

print("Identified Topics:\n")
for topic_id, topic in topics:
    print(f"Topic #{topic_id + 1}: {', '.join([word for word, prob in topic])}")

In [ ]:
# Visualise LDA Topics
vis_data = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary, mds='mmds')
pyLDAvis.display(vis_data)